# Feature Engineering & Training-Budget Ablation

This notebook continues from a LightGBM baseline (5-fold stratified CV, OOF balanced
accuracy **0.9389 +/- 0.0012**, public LB **0.94051**) built on the raw feature set.
Here we run two follow-up experiments before trying a different model family:

1. A training-budget ablation on the baseline's feature set -- none of the baseline's
   folds actually early-stopped, so we check whether more rounds / a lower learning
   rate buys additional CV.
2. A root-cause investigation into `sleep_duration`'s surprisingly high (#2) feature
   importance, using binned and interaction views instead of a marginal histogram.
3. A round of engineered features -- missingness indicators, categorical interaction
   crosses, and out-of-fold smoothed multiclass target encoding -- retrained with the
   tuned budget from step 1.
4. A feature-importance check on the engineered model.
5. A candidate `submission.csv` (not auto-submitted from this notebook).

Spoiler: both the budget ablation and the engineered-feature set underperform the
original baseline on cross-validation. We keep the results here because a negative
result -- and the reasoning behind it -- is worth documenting.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

import os, glob

DATA_DIR = Path('/kaggle/input/playground-series-s6e7')
if not (DATA_DIR / 'train.csv').exists():
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if hits:
        DATA_DIR = Path(hits[0]).parent
    else:
        DATA_DIR = Path('..') / 'data'  # local fallback for testing outside Kaggle

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_DIR
TARGET = 'health_condition'
NUMERIC_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
                 'step_count', 'exercise_duration', 'water_intake']
CATEGORICAL_COLS = ['diet_type', 'stress_level', 'sleep_quality',
                     'physical_activity_level', 'smoking_alcohol', 'gender']

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

for col in CATEGORICAL_COLS:
    train[col] = train[col].fillna('missing').astype(str)
    test[col] = test[col].fillna('missing').astype(str)
    categories = sorted(set(train[col].unique()) | set(test[col].unique()))
    train[col] = pd.Categorical(train[col], categories=categories)
    test[col] = pd.Categorical(test[col], categories=categories)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train[TARGET])
n_classes = len(label_encoder.classes_)
print('classes:', list(label_encoder.classes_))

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))
print('train', train.shape, 'test', test.shape)

classes: ['at-risk', 'fit', 'unhealthy']
train (690088, 15) test (295753, 14)


## Section A -- Training-budget ablation

The baseline model used `n_estimators=2000, learning_rate=0.05`, and **none of its
folds early-stopped** -- validation loss was still improving at round 2000. Here we
re-run the identical baseline feature set with a bigger round budget and a lower
learning rate, to see how much CV improves from training budget alone before adding
any new features. Whichever configuration wins becomes the shared hyperparameter
budget for the engineered-feature run in Section D, so that comparison isn't
confounded by a simultaneous training-budget change and feature-set change.

In [2]:
from tqdm.auto import tqdm

class TqdmCallback:
    """LightGBM training callback that drives a tqdm bar, one tick per boosting round."""
    def __init__(self, total, desc):
        self.pbar = tqdm(total=total, desc=desc, leave=False)
        self.order = 10
    def __call__(self, env):
        self.pbar.update(1)

BASE_FEATURES = NUMERIC_COLS + CATEGORICAL_COLS

def run_cv(feature_frame, test_frame, categorical_features, lgb_params, folds, y, n_classes, desc='folds'):
    oof_pred = np.zeros(len(feature_frame), dtype=int)
    test_proba = np.zeros((len(test_frame), n_classes))
    fold_scores, best_iters, models = [], [], []
    for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc=desc)):
        X_tr, X_val = feature_frame.iloc[tr_idx], feature_frame.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = lgb.LGBMClassifier(objective='multiclass', num_class=n_classes,
                                    class_weight='balanced', n_jobs=-1, random_state=42,
                                    verbosity=-1, **lgb_params)
        round_pbar = TqdmCallback(lgb_params['n_estimators'], f'{desc} fold {fold} rounds')
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='multi_logloss',
                  categorical_feature=categorical_features,
                  callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False),
                             lgb.log_evaluation(0), round_pbar])
        round_pbar.pbar.close()
        val_pred = model.predict_proba(X_val, num_iteration=model.best_iteration_).argmax(axis=1)
        oof_pred[val_idx] = val_pred
        fold_scores.append(balanced_accuracy_score(y_val, val_pred))
        best_iters.append(model.best_iteration_)
        test_proba += model.predict_proba(test_frame, num_iteration=model.best_iteration_) / len(folds)
        models.append(model)
    oof_bal_acc = balanced_accuracy_score(y, oof_pred)
    return {'oof_pred': oof_pred, 'test_proba': test_proba, 'fold_scores': fold_scores,
            'best_iters': best_iters, 'oof_balanced_accuracy': oof_bal_acc, 'models': models}

budget_params = dict(n_estimators=5000, learning_rate=0.03, num_leaves=63,
                      subsample=0.8, colsample_bytree=0.8)

result_a = run_cv(train[BASE_FEATURES], test[BASE_FEATURES], CATEGORICAL_COLS,
                   budget_params, folds, y, n_classes, desc='Section A')

print('best_iterations:', result_a['best_iters'])
print('fold scores:', [round(s, 4) for s in result_a['fold_scores']])
print(f"Section A OOF balanced accuracy: {result_a['oof_balanced_accuracy']:.4f}")
print(f"v0.1 OOF balanced accuracy:      0.9389 (baseline, n_estimators=2000, lr=0.05)")

Section A:   0%|          | 0/5 [00:00<?, ?it/s]

Section A fold 0 rounds:   0%|          | 0/5000 [00:00<?, ?it/s]

Section A fold 1 rounds:   0%|          | 0/5000 [00:00<?, ?it/s]

Section A fold 2 rounds:   0%|          | 0/5000 [00:00<?, ?it/s]

Section A fold 3 rounds:   0%|          | 0/5000 [00:00<?, ?it/s]

Section A fold 4 rounds:   0%|          | 0/5000 [00:00<?, ?it/s]

best_iterations: [5000, 5000, 5000, 5000, 4998]
fold scores: [0.9283, 0.9297, 0.9316, 0.9282, 0.9275]
Section A OOF balanced accuracy: 0.9290
v0.1 OOF balanced accuracy:      0.9389 (baseline, n_estimators=2000, lr=0.05)


## Section B -- Root-causing `sleep_duration`'s #2 feature importance

The baseline's univariate histograms (split by class) showed heavy overlap for
`sleep_duration`, which made its #2 feature-importance ranking puzzling. We look at a
binned/quantile view, plus a 2D cut against `stress_level` (the #1 feature), to see
whether the signal is a threshold effect or an interaction that a marginal histogram
can't surface.

In [3]:
train['sleep_duration_bin'], sleep_bin_edges = pd.qcut(
    train['sleep_duration'], q=10, duplicates='drop', retbins=True)

print('--- health_condition share by sleep_duration decile (train) ---')
display(pd.crosstab(train['sleep_duration_bin'], train[TARGET], normalize='index'))

print('\n--- at-risk share by sleep_duration decile x stress_level ---')
pivot = train.pivot_table(index='sleep_duration_bin', columns='stress_level',
                           values=TARGET, aggfunc=lambda s: (s == 'at-risk').mean())
display(pivot)

--- health_condition share by sleep_duration decile (train) ---


health_condition,at-risk,fit,unhealthy
sleep_duration_bin,,,
"(2.999, 5.47]",0.603660,0.002212,0.394127
"(5.47, 5.92]",0.625714,0.002153,0.372133
"(5.92, 6.37]",0.951131,0.003254,0.045616
"(6.37, 6.7]",0.995944,0.002557,0.001499
"(6.7, 6.99]",0.990852,0.007499,0.001648
"(6.99, 7.28]",0.910181,0.086218,0.003601
"(7.28, 7.6]",0.868123,0.129332,0.002545
"(7.6, 8.04]",0.883261,0.114399,0.002340
"(8.04, 8.55]",0.887977,0.109756,0.002267



--- at-risk share by sleep_duration decile x stress_level ---


stress_level,high,low,medium,missing
sleep_duration_bin,,,,
"(2.999, 5.47]",0.001968,0.994349,0.992970,0.612671
"(5.47, 5.92]",0.005078,0.995603,0.993138,0.620376
"(5.92, 6.37]",0.829981,0.994065,0.995031,0.951351
"(6.37, 6.7]",0.996662,0.995493,0.995888,0.995680
"(6.7, 6.99]",0.992293,0.984781,0.993991,0.990109
"(6.99, 7.28]",0.995146,0.685188,0.992412,0.911546
"(7.28, 7.6]",0.995115,0.573540,0.994073,0.871573
"(7.6, 8.04]",0.995141,0.616512,0.994108,0.882234
"(8.04, 8.55]",0.995863,0.629613,0.994028,0.894638


## Section C -- Engineered features

- Missingness indicators for numeric columns only (categorical `NaN`s already have an
  explicit `"missing"` level from the baseline's preprocessing, so an indicator there
  would be redundant).
- Categorical x categorical interactions: `stress_level` x `physical_activity_level`
  (the two dominant EDA signals) and `sleep_quality` x `smoking_alcohol` (the two
  secondary signals).
- `sleep_duration` decile (from Section B, train-derived bin edges applied to test --
  no target used, so no leakage) x `stress_level`.
- Out-of-fold smoothed multiclass target encoding for `stress_level`,
  `physical_activity_level`, and the two interaction crosses -- fit within each
  training fold only, with the test encoding averaged across the 5 folds' fitted maps.

In [4]:
test_sleep_clipped = test['sleep_duration'].clip(lower=sleep_bin_edges[0], upper=sleep_bin_edges[-1])
test['sleep_duration_bin'] = pd.cut(test_sleep_clipped, bins=sleep_bin_edges, include_lowest=True)

# pd.cut on test produces NaN both for genuinely-missing sleep_duration AND for values
# outside train's bin range (a real bug hit on the first run: astype(str) on the
# Categorical(Interval) column doesn't stringify those NaNs, so string concatenation in
# make_cross() mixed float NaN with str and broke sorted(set(...))). Clipping test into
# train's range before cutting removes the out-of-range source; the null mask on the
# original numeric column then correctly captures only genuine missingness.
train['sleep_duration_bin'] = train['sleep_duration_bin'].astype(str)
train.loc[train['sleep_duration'].isnull(), 'sleep_duration_bin'] = 'missing'
test['sleep_duration_bin'] = test['sleep_duration_bin'].astype(str)
test.loc[test['sleep_duration'].isnull(), 'sleep_duration_bin'] = 'missing'

MISSING_INDICATOR_COLS = []
for col in NUMERIC_COLS:
    ind_col = f'is_missing_{col}'
    train[ind_col] = train[col].isnull().astype(int)
    test[ind_col] = test[col].isnull().astype(int)
    MISSING_INDICATOR_COLS.append(ind_col)

def make_cross(df, col_a, col_b, name):
    df[name] = df[col_a].astype(str) + '_' + df[col_b].astype(str)

make_cross(train, 'stress_level', 'physical_activity_level', 'stress_x_activity')
make_cross(test, 'stress_level', 'physical_activity_level', 'stress_x_activity')
make_cross(train, 'sleep_quality', 'smoking_alcohol', 'sleepq_x_smoking')
make_cross(test, 'sleep_quality', 'smoking_alcohol', 'sleepq_x_smoking')
make_cross(train, 'sleep_duration_bin', 'stress_level', 'sleepbin_x_stress')
make_cross(test, 'sleep_duration_bin', 'stress_level', 'sleepbin_x_stress')

CROSS_COLS = ['stress_x_activity', 'sleepq_x_smoking', 'sleepbin_x_stress']
for col in CROSS_COLS:
    categories = sorted(set(train[col].unique()) | set(test[col].unique()))
    train[col] = pd.Categorical(train[col], categories=categories)
    test[col] = pd.Categorical(test[col], categories=categories)

print('cross feature cardinalities:', {c: train[c].nunique() for c in CROSS_COLS})

cross feature cardinalities: {'stress_x_activity': 16, 'sleepq_x_smoking': 16, 'sleepbin_x_stress': 44}


In [5]:
def oof_target_encode(train_col, test_col, y, n_classes, folds, alpha=20):
    """Smoothed multiclass target encoding, fit per-fold to avoid leakage.
    Returns (n_train, n_classes) OOF-encoded array and (n_test, n_classes) test array
    (test array = mean over the 5 folds' fitted encodings)."""
    n_train = len(train_col)
    oof_encoded = np.zeros((n_train, n_classes))
    test_encoded = np.zeros((len(test_col), n_classes))
    class_cols = [f'k{i}' for i in range(n_classes)]
    onehot = pd.get_dummies(pd.Series(y)).values

    for tr_idx, val_idx in folds:
        cats_tr = pd.Series(train_col.iloc[tr_idx].astype(str).values)
        y_tr_onehot = onehot[tr_idx]
        prior = pd.Series(y_tr_onehot.mean(axis=0), index=class_cols)

        stats = pd.DataFrame(y_tr_onehot, columns=class_cols)
        stats['cat'] = cats_tr.values
        grp_sum = stats.groupby('cat')[class_cols].sum()
        grp_count = stats.groupby('cat').size()
        enc_map = grp_sum.add(alpha * prior, axis=1).div(grp_count + alpha, axis=0)

        val_cats = pd.Series(train_col.iloc[val_idx].astype(str).values)
        oof_encoded[val_idx] = enc_map.reindex(val_cats).fillna(prior).values

        test_cats = pd.Series(test_col.astype(str).values)
        test_encoded += enc_map.reindex(test_cats).fillna(prior).values / len(folds)

    return oof_encoded, test_encoded

TARGET_ENCODE_COLS = ['stress_level', 'physical_activity_level', 'stress_x_activity', 'sleepq_x_smoking']
TARGET_ENCODED_FEATURE_COLS = []
for col in tqdm(TARGET_ENCODE_COLS, desc='target encoding'):
    oof_enc, test_enc = oof_target_encode(train[col], test[col], y, n_classes, folds)
    for k in range(n_classes):
        fcol = f'te_{col}_k{k}'
        train[fcol] = oof_enc[:, k]
        test[fcol] = test_enc[:, k]
        TARGET_ENCODED_FEATURE_COLS.append(fcol)

print(f'{len(TARGET_ENCODED_FEATURE_COLS)} target-encoded columns added')
train[TARGET_ENCODED_FEATURE_COLS].head()

target encoding:   0%|          | 0/4 [00:00<?, ?it/s]

12 target-encoded columns added


,te_stress_level_k0,te_stress_level_k1,te_stress_level_k2,te_physical_activity_level_k0,te_physical_activity_level_k1,te_physical_activity_level_k2,te_stress_x_activity_k0,te_stress_x_activity_k1,te_stress_x_activity_k2,te_sleepq_x_smoking_k0,te_sleepq_x_smoking_k1,te_sleepq_x_smoking_k2
0,0.718177,0.003580,0.278243,0.916978,0.002546,0.080476,0.730100,0.003004,0.266896,0.852269,0.036985,0.110745
1,0.796589,0.200656,0.002756,0.911725,0.003151,0.085125,0.994866,0.002655,0.002478,0.849792,0.037452,0.112756
2,0.717987,0.003477,0.278536,0.742793,0.172012,0.085195,0.701104,0.004152,0.294744,0.801681,0.021922,0.176397
3,0.717251,0.003549,0.279200,0.742734,0.171952,0.085314,0.699878,0.004492,0.295630,0.857525,0.058346,0.084130
4,0.859829,0.057284,0.082887,0.916992,0.002442,0.080566,0.919120,0.002683,0.078197,0.862237,0.053248,0.084515


## Section D — Retrain with engineered features

In [6]:
ENGINEERED_CATEGORICAL_COLS = CATEGORICAL_COLS + CROSS_COLS
ENGINEERED_FEATURES = (NUMERIC_COLS + CATEGORICAL_COLS + MISSING_INDICATOR_COLS
                        + CROSS_COLS + TARGET_ENCODED_FEATURE_COLS)

result_d = run_cv(train[ENGINEERED_FEATURES], test[ENGINEERED_FEATURES],
                   ENGINEERED_CATEGORICAL_COLS, budget_params, folds, y, n_classes, desc='Section D')

print('best_iterations:', result_d['best_iters'])
print('fold scores:', [round(s, 4) for s in result_d['fold_scores']])
print(f"Section D OOF balanced accuracy: {result_d['oof_balanced_accuracy']:.4f}")
print(f"Section A OOF balanced accuracy: {result_a['oof_balanced_accuracy']:.4f} (same budget, v0.1 features only)")
print(f"v0.1 OOF balanced accuracy:      0.9389 (n_estimators=2000, lr=0.05)")
print()
print(classification_report(y, result_d['oof_pred'], target_names=label_encoder.classes_, digits=4))

Section D:   0%|          | 0/5 [00:00<?, ?it/s]

Section D fold 0 rounds:   0%|          | 0/5000 [00:00<?, ?it/s]

Section D fold 1 rounds:   0%|          | 0/5000 [00:00<?, ?it/s]

Section D fold 2 rounds:   0%|          | 0/5000 [00:00<?, ?it/s]

Section D fold 3 rounds:   0%|          | 0/5000 [00:00<?, ?it/s]

Section D fold 4 rounds:   0%|          | 0/5000 [00:00<?, ?it/s]

best_iterations: [4987, 4854, 5000, 4760, 4661]
fold scores: [0.9279, 0.9256, 0.9277, 0.9225, 0.9239]
Section D OOF balanced accuracy: 0.9255
Section A OOF balanced accuracy: 0.9290 (same budget, v0.1 features only)
v0.1 OOF balanced accuracy:      0.9389 (n_estimators=2000, lr=0.05)

              precision    recall  f1-score   support

     at-risk     0.9846    0.9664    0.9754    592561
         fit     0.8287    0.9021    0.8638     39803
   unhealthy     0.8040    0.9082    0.8529     57724

    accuracy                         0.9578    690088
   macro avg     0.8725    0.9255    0.8974    690088
weighted avg     0.9605    0.9578    0.9587    690088



## Section E — Feature importance

In [7]:
importances = pd.DataFrame({
    'feature': ENGINEERED_FEATURES,
    'gain_importance': np.mean([m.booster_.feature_importance(importance_type='gain')
                                 for m in result_d['models']], axis=0),
}).sort_values('gain_importance', ascending=False)
importances

,feature,gain_importance
22,sleepbin_x_stress,9.007233e+06
30,te_stress_x_activity_k1,4.412651e+06
0,sleep_duration,1.886471e+06
29,te_stress_x_activity_k0,9.629582e+05
20,stress_x_activity,8.703207e+05
2,bmi,5.557575e+05
3,calorie_expenditure,2.985940e+05
4,step_count,2.969963e+05
6,water_intake,2.903555e+05
5,exercise_duration,2.833977e+05


## Section F — Candidate submission

In [8]:
best_result = result_d if result_d['oof_balanced_accuracy'] >= result_a['oof_balanced_accuracy'] else result_a
best_label = 'Section D (engineered features)' if best_result is result_d else 'Section A (budget-only, v0.1 features)'
print(f'Best config: {best_label}, OOF balanced accuracy {best_result["oof_balanced_accuracy"]:.4f}')

test_pred = best_result['test_proba'].argmax(axis=1)
submission = pd.DataFrame({
    'id': test['id'],
    TARGET: label_encoder.inverse_transform(test_pred),
})

sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
assert list(submission.columns) == list(sample_submission.columns)
assert len(submission) == len(sample_submission)
assert set(submission[TARGET].unique()) <= set(label_encoder.classes_)
assert submission[TARGET].isnull().sum() == 0

submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
print(f'wrote {submission_path} ({len(submission)} rows) — NOT auto-submitted to Kaggle')
submission[TARGET].value_counts(normalize=True)

Best config: Section A (budget-only, v0.1 features), OOF balanced accuracy 0.9290
wrote ../data/submission.csv (295753 rows) — NOT auto-submitted to Kaggle


health_condition
at-risk      0.840265
unhealthy    0.095289
fit          0.064446
Name: proportion, dtype: float64

## Summary

**Negative result: neither experiment beat the baseline (OOF 0.9389, LB 0.94051). The
original baseline remains the best model.**

- **Section A (training-budget ablation)**: OOF **0.9290** with `n_estimators=5000,
  learning_rate=0.03` (vs. the baseline's `2000`/`0.05`) on the identical feature set --
  *worse* than the baseline. 4/5 folds ran the full 5000 rounds without early
  stopping. Early stopping tracked `multi_logloss`, not balanced accuracy, so pushing
  training further with a lower learning rate kept improving logloss while the
  decision boundary drifted away from balanced per-class recall. The lesson: the
  baseline's original configuration was already closer to optimal for the actual
  competition metric than the "no fold early-stopped" observation suggested.
- **Section B (`sleep_duration` root-cause)**: confirmed a genuine, strong
  non-monotonic signal. Short sleep (<6h) -> ~60% at-risk / ~37% unhealthy (vs. ~8%
  baseline). Mid-range sleep (6-8.5h) -> ~90-99% at-risk, near-zero unhealthy/fit.
  Longer sleep -> `fit` share up to ~11-13%. Strong interaction with `stress_level`:
  at `stress_level=low`, at-risk share drops sharply as sleep increases (favoring
  `fit`); at `stress_level=high`, at-risk stays ~99.5% regardless of sleep except at
  very short sleep, where it collapses toward `unhealthy`. This fully explains
  `sleep_duration`'s #2 importance ranking in the baseline -- a univariate histogram
  just can't show a non-monotonic, interaction-dependent effect like this.
- **Section D (engineered features)**: missingness indicators, `stress_level` x
  `physical_activity_level` and `sleep_quality` x `smoking_alcohol` crosses,
  `sleep_duration`-decile x `stress_level` cross, and OOF smoothed multiclass target
  encoding (12 columns). OOF **0.9255** -- worse than *both* the baseline and Section
  A. Per-class recall shifted rather than improved: `at-risk` 0.956 -> 0.966, but
  `fit` 0.929 -> 0.902 and `unhealthy` 0.932 -> 0.908 -- net negative for balanced
  accuracy.
- **Feature importance on Section D**: `sleepbin_x_stress` (the engineered cross)
  became the single dominant feature (9.0M gain, more than double the baseline's raw
  `stress_level` at 4.59M) -- confirming the Section B interaction hypothesis at the
  model level -- while raw `stress_level` collapsed to near-zero importance (99.7),
  fully absorbed into the cross feature. **The interaction is genuinely informative,
  but the larger 35-feature set (vs. the baseline's 13) still net-hurt CV**, likely
  from added variance: Section D's per-fold `best_iteration` varied widely
  (4661-5000) vs. Section A's near-uniform ~5000, suggesting less stable training.
- **Distinguishing an informative feature from a better model**: this is the core
  takeaway. `sleepbin_x_stress` is unambiguously the most important single feature in
  the Section D model, yet the model it belongs to is worse than one without it (or
  its underlying raw columns) at this budget/regularization setting. Feature
  importance ranks *within* a fitted model; it says nothing about whether including
  the feature made that model better than the alternative.
- The written `submission.csv` uses Section A's model (0.9290 > Section D's 0.9255,
  correctly selected by the notebook's own comparison logic), but this is still worse
  than the original baseline's already-submitted result.

**Takeaway**: the original baseline remains the model to beat. The natural next steps
are a different model family (e.g. CatBoost, which may handle the interaction/encoding
signal with less overfitting via ordered boosting) and per-class threshold tuning on
the baseline's OOF predictions rather than these engineered features, since the
baseline is still the stronger base model.